In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
df = pd.read_csv('../03_telco-churn/telco-churn.csv')
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [3]:
df.columns = df.columns.str.lower()
categorical_columns = df.select_dtypes(include='string').columns.to_list()
df[categorical_columns] = df[categorical_columns].apply(lambda x: x.str.lower().str.replace(' ', '_', regex=False))
df['totalcharges'] = df['totalcharges'].apply(pd.to_numeric, errors='coerce')
df['seniorcitizen'] = df['seniorcitizen'].astype('str')
df['totalcharges'] = df['totalcharges'].fillna(0)
df['churn'] = (df.churn == 'yes').astype('int')

In [4]:
from sklearn.model_selection import train_test_split

df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=1)

y_train = df_train.churn
y_val = df_val.churn
y_test = df_test.churn

del df_train['churn']
del df_val['churn']
del df_test['churn']

In [5]:
numerical_columns = df_train_full.columns[(df_train_full.dtypes=='float') | (df_train_full.dtypes=='int')].tolist()
numerical_columns.remove('churn')

categorical_columns = df_train_full.columns[df_train_full.dtypes=='str'].tolist()
categorical_columns.remove('customerid')

In [6]:
y_train_full = df_train_full.churn.values
df_train_one_hot_encoded = pd.get_dummies(df_train_full[categorical_columns], dtype=int)
df_train_full = pd.concat([df_train_full[numerical_columns], df_train_one_hot_encoded], axis=1)

df_train_one_hot_encoded = pd.get_dummies(df_train[categorical_columns], dtype=int)
df_train = pd.concat([df_train[numerical_columns], df_train_one_hot_encoded], axis=1)

df_val_one_hot_encoded = pd.get_dummies(df_val[categorical_columns], dtype=int)
df_val = pd.concat([df_val[numerical_columns], df_val_one_hot_encoded], axis=1)

df_test_one_hot_encoded = pd.get_dummies(df_test[categorical_columns], dtype=int)
df_test = pd.concat([df_test[numerical_columns], df_test_one_hot_encoded], axis=1)

In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=100)
model.fit(df_train_full, y_train_full)

y_pred = model.predict(df_val)

from sklearn.metrics import accuracy_score

print(accuracy_score(y_val, y_pred)) # accuracy score is misleading when there is class imbalance

0.8055358410220014


/Users/bioinfo/Projects/applied-data/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
scores = cross_val_score(model, df_train_full, y_train_full, cv=cv, scoring='roc_auc')

print(scores.mean())
print(scores.std())

0.8406124146274744
0.005371491344904595


/Users/bioinfo/Projects/applied-data/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/bioinfo/Projects/applied-data/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-

In [9]:
import pickle

with open('model.bin', 'wb') as file:
    pickle.dump(model, file)

In [10]:
import pickle

with open('model.bin', 'rb') as file:
    pretrained_model = pickle.load(file)

In [11]:
customer = {
    'customerid': '8879-zkjof',
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'no',
    'dependents': 'no',
    'tenure': 41,
    'phoneservice': 'yes',
    'multiplelines': 'no',
    'internetservice': 'dsl',
    'onlinesecurity': 'yes',
    'onlinebackup': 'no',
    'deviceprotection': 'yes',
    'techsupport': 'yes',
    'streamingtv': 'yes',
    'streamingmovies': 'yes',
    'contract': 'one_year',
    'paperlessbilling': 'yes',
    'paymentmethod': 'bank_transfer_(automatic)',
    'monthlycharges': 79.85,
    'totalcharges': 3320.75
}

In [12]:
def prepare_data(customer_dict):
    df_small = pd.DataFrame([customer_dict])
    df_small['seniorcitizen'] = df_small['seniorcitizen'].astype('str')

    df_small_one_hot_encoded = pd.get_dummies(df_small[categorical_columns], dtype=int)
    df_final = pd.concat([df_small[numerical_columns], df_small_one_hot_encoded], axis=1)
    df_final = df_final.reindex(columns=df_train_full.columns, fill_value=0)

    return df_final

In [13]:
prepare_data(customer).head()

,tenure,monthlycharges,totalcharges,gender_female,gender_male,seniorcitizen_0,seniorcitizen_1,partner_no,partner_yes,dependents_no,...,streamingmovies_yes,contract_month-to-month,contract_one_year,contract_two_year,paperlessbilling_no,paperlessbilling_yes,paymentmethod_bank_transfer_(automatic),paymentmethod_credit_card_(automatic),paymentmethod_electronic_check,paymentmethod_mailed_check
0,41,79.85,3320.75,1,0,1,0,1,0,1,...,1,0,1,0,0,1,1,0,0,0


In [14]:
customer_prepared = prepare_data(customer)
pretrained_model.predict_proba(customer_prepared)[:, 1]

array([0.05703756])